# Problem 5: Document Question Answering

**Objective:** build a web application that answers questions using *only the content of an uploaded document*, and cites the page on which each answer was found.

AI models do not have access to an institution's own documents, such as course syllabi, hostel regulations or student handbooks. When asked about them, a model may produce answers that sound convincing but are false. The established solution is **Retrieval-Augmented Generation (RAG)**:

```
 PDF document ──► 1. split into short chunks ──► 2. convert each chunk into numbers (an "embedding")
                                                           │
 Question ──► 3. find the chunks whose embeddings are closest to the question's embedding
                                                           │
              4. give ONLY those chunks to the AI ──► answer with page citations
```

Two **open-weight models** are used: `all-MiniLM-L6-v2`, which creates embeddings and runs free of charge within Colab, and **gpt-oss** via Groq, which writes the answer.

**Learning outcomes:** embeddings, semantic search, and methods for reducing inaccurate AI output.

### How to use this notebook
- A notebook is made up of **cells**. To run a cell, select it and press **Shift + Enter** (or select the run button).
- On the first run, run the cells **in order, from top to bottom**.
- Comments marked **STUDENT TODO** show where changes are expected:
  - **Level 1:** text changes. No coding experience required.
  - **Level 2:** changes to settings and values.
  - **Level 3:** new features. Mentors are available to help.
- After changing a cell, **re-run that cell and then re-run the "Launch the application" cell** for the change to take effect.
- If the notebook stops working as expected, select **Runtime → Restart session** and run all cells again from the top.

### Obtaining a free Groq API key (one per team; about 2 minutes)

Groq is a free service that hosts **open-weight AI models**, such as OpenAI's gpt-oss. The code in this notebook sends a message to Groq and receives the model's response.

1. Go to **https://console.groq.com** and sign in with a Google or GitHub account.
2. Select **API Keys** → **Create API Key**, enter any name, and **copy** the key (it begins with `gsk_`).
3. In Colab, select the **key icon** in the left sidebar (**Secrets**).
4. Select **Add new secret**. Set **Name** to `GROQ_API_KEY` and **Value** to your key. Enable **Notebook access**.

> **Important:** do not paste the key directly into a code cell. Anyone the notebook is shared with would be able to see it. The Secrets panel keeps it private.
>
> **No key yet:** the notebook runs in **demonstration mode** with sample responses, so you can explore and edit it before a key is available.

> **Note:** in demonstration mode, the retrieval stages (steps 1–3) still run normally. The application shows the passages found, without an AI-generated answer.

## Step 0: Installation and setup (run all three cells)

In [ ]:
# Installs the required libraries. This takes about one minute.
!pip install -q "gradio>=5,<6" openai pypdf sentence-transformers

In [ ]:
# SETUP: run this cell once. No changes are required here.
import os
import openai
from openai import OpenAI

def _get_secret(name):
    """Reads a secret from the Colab Secrets panel (or from an environment variable)."""
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

GROQ_API_KEY = _get_secret("GROQ_API_KEY")
DEMO_MODE = not GROQ_API_KEY

# Groq uses the same interface as the OpenAI API, so the openai library is used
# and pointed at Groq's servers.
client = None if DEMO_MODE else OpenAI(api_key=GROQ_API_KEY, base_url="https://api.groq.com/openai/v1")

def _friendly_error(e):
    if isinstance(e, openai.AuthenticationError):
        return "Error: the GROQ_API_KEY was rejected. Check that the complete key (beginning with gsk_) is saved in Colab Secrets, then re-run the setup cell."
    if isinstance(e, openai.RateLimitError):
        return "Error: the Groq free-tier usage limit has been reached. Wait about one minute and try again, or set MODEL = 'openai/gpt-oss-20b'."
    if isinstance(e, openai.NotFoundError):
        return f"Error: Groq does not recognise the model '{MODEL}'. Check the spelling; the model may also have been retired. See https://console.groq.com/docs/models"
    if isinstance(e, openai.APIConnectionError):
        return "Error: unable to connect to Groq. Check the internet connection and try again."
    return f"Error: the request to the AI service failed ({e})."

def ask_llm(messages, temperature=0.7, json_mode=False, demo_reply="(Demonstration mode response)"):
    """Sends a list of chat messages to the AI model and returns its response as text.
    In demonstration mode, returns `demo_reply` instead. If the request fails,
    returns a message beginning with "Error:" instead of stopping the program."""
    if DEMO_MODE:
        return demo_reply
    try:
        extra = {"response_format": {"type": "json_object"}} if json_mode else {}
        if MODEL.startswith("openai/gpt-oss"):
            extra["reasoning_effort"] = "low"  # gpt-oss models reason before answering; "low" keeps responses fast
        response = client.chat.completions.create(
            model=MODEL, messages=messages, temperature=temperature, **extra
        )
        return response.choices[0].message.content
    except Exception as e:
        return _friendly_error(e)

if DEMO_MODE:
    print("Demonstration mode: no GROQ_API_KEY was found, so the application will show sample responses.")
    print("To connect the AI model: Secrets (left sidebar) → add GROQ_API_KEY → enable Notebook access → re-run this cell.")
else:
    print("Connected to Groq. The AI model is ready.")

In [ ]:
# Downloads the open-weight embedding model (about 90 MB). It converts text into lists of 384 numbers.
from sentence_transformers import SentenceTransformer

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"  # STUDENT TODO (Level 3): alternative: "BAAI/bge-small-en-v1.5"
embedder = SentenceTransformer(EMBED_MODEL_NAME)
print(f"Loaded embedding model: {EMBED_MODEL_NAME}")

## Step 1: Settings

These values control **how the document is divided and searched**. Changing them affects the answers, which makes them a suitable subject for an experiment in the demonstration.

In [ ]:
# STUDENT TODO (Level 1): set the application name and description.
APP_TITLE = "Document Question Answering"
APP_DESCRIPTION = "Upload a PDF and ask questions. Answers are based only on the document and include page references."

# STUDENT TODO (Level 2): the number of words in each chunk.
#   Small chunks (50)   more precise matches, but may lose context
#   Large chunks (300)  more context, but less precise and more text for the AI to process
CHUNK_SIZE = 120

# STUDENT TODO (Level 2): the number of words shared by neighbouring chunks, so sentences are not split.
CHUNK_OVERLAP = 30

# STUDENT TODO (Level 2): the number of chunks given to the AI for each question.
TOP_K = 3

# STUDENT TODO (Level 2): the rules for the AI. The rule to use only the context is what prevents invented answers.
ANSWER_RULES = """
You answer questions about a document using ONLY the context passages provided.
- Each passage begins with its page number, for example [Page 2].
- After each fact you use, cite the page in brackets, for example (p. 2).
- If the answer is not in the context, reply exactly: "The answer could not be found in the document."
- Keep answers concise: 2 to 4 sentences.
"""

TEMPERATURE = 0.2  # a low value keeps answers close to the document

# STUDENT TODO (Level 2): select the open-weight model that generates responses.
#   "openai/gpt-oss-20b"   OpenAI gpt-oss, 20 billion parameters. Fast and suitable for most tasks. (default)
#   "openai/gpt-oss-120b"  OpenAI gpt-oss, 120 billion parameters. More capable, slower, lower limits.
#   "qwen/qwen3.8-27b"     Alibaba Qwen (a preview model, which Groq may withdraw at short notice).
#   Groq retires models from time to time. Current list: https://console.groq.com/docs/models
MODEL = "openai/gpt-oss-20b"

print("Settings configured.")

## Step 2: Sample document

So that the application works immediately, a fictional course syllabus is provided. Each item in the list represents one page. A real PDF can be uploaded in the application later.

In [ ]:
# STUDENT TODO (Level 1): this text may be edited, or a real PDF may be uploaded in the application instead.
SAMPLE_PAGES = [
    """CS 101: Introduction to Programming (Sample Syllabus). Instructor: Dr. A. Rao. Email: arao@example.edu.
    Lectures are held Monday and Wednesday, 10:00 to 11:30 AM, in Room B-204. Labs are on Friday from 2:00 to 4:00 PM
    in the Computer Lab (Room C-110). Office hours: Tuesday 3:00 to 5:00 PM, or by appointment. This course introduces
    programming using Python. No prior coding experience is required.""",

    """Grading. Weekly lab exercises: 20%. Two mini-projects: 30% (15% each). Midterm exam: 20%. Final exam: 25%.
    Class participation: 5%. A score of 90% or above earns an A, 80 to 89% a B, 70 to 79% a C, and 60 to 69% a D.
    Late submissions lose 10% per day, up to a maximum of 3 days, after which they receive zero. Each student has
    two grace days per semester that may be used to submit any lab late without penalty.""",

    """Schedule. Weeks 1-2: variables, data types, and input/output. Weeks 3-4: conditions and loops. Week 5: functions.
    Week 6: lists and dictionaries. Week 7: Midterm exam (in class, closed book). Weeks 8-9: files and error handling.
    Weeks 10-11: an introduction to APIs and working with data. Weeks 12-13: mini-project 2, a team project of up to
    3 students. Week 14: review. The final exam is a 2-hour written exam during the examination period.""",

    """Policies. Attendance: students must attend at least 75% of lectures to be eligible for the final exam.
    Academic integrity: students may discuss ideas with classmates, but all submitted code must be their own. AI
    tools may be used for learning and debugging, but their use must be stated in a comment at the top of the file.
    Laptops: bring a laptop to every lab. Students without a laptop may reserve a machine in the Computer Lab.
    Textbook: "Automate the Boring Stuff with Python" (available free online).""",
]
print(f"Sample document contains {len(SAMPLE_PAGES)} pages.")

## Step 3: The RAG pipeline (run this cell)

The comments describe each stage of the pipeline. This cell is the core of the project.

In [ ]:
import numpy as np
from pypdf import PdfReader

knowledge_base = {"name": None, "chunks": [], "vectors": None}  # populated by build_knowledge_base()


def read_pdf(path):
    """Returns a list of page texts from a PDF file."""
    return [page.extract_text() or "" for page in PdfReader(path).pages]


def split_into_chunks(pages):
    """STAGE 1: split each page into overlapping chunks of about CHUNK_SIZE words."""
    chunks = []
    step = max(1, CHUNK_SIZE - CHUNK_OVERLAP)
    for page_number, text in enumerate(pages, start=1):
        words = text.split()
        for start in range(0, len(words), step):
            chunk_words = words[start:start + CHUNK_SIZE]
            if len(chunk_words) > 5:  # skip very short fragments
                chunks.append({"page": page_number, "text": " ".join(chunk_words)})
            if start + CHUNK_SIZE >= len(words):
                break
    return chunks


def build_knowledge_base(pages, name):
    """STAGE 2: convert each chunk into an embedding (a list of numbers representing its meaning)."""
    chunks = split_into_chunks(pages)
    if not chunks:
        raise ValueError("No text was found. The PDF may contain scanned images only. Try another file.")
    vectors = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True)
    knowledge_base.update(name=name, chunks=chunks, vectors=vectors)
    return f"Loaded **{name}**: {len(pages)} pages, {len(chunks)} chunks. Ready for questions."


def search(question, top_k=TOP_K):
    """STAGE 3: find the chunks whose meaning is closest to the question."""
    question_vector = embedder.encode([question], normalize_embeddings=True)[0]
    similarities = knowledge_base["vectors"] @ question_vector   # similarity of each chunk (1.0 = identical)
    best = np.argsort(similarities)[::-1][:top_k]
    return [{**knowledge_base["chunks"][i], "score": float(similarities[i])} for i in best]


def answer(question):
    """STAGE 4: give the most relevant chunks to the AI and ask it to answer using only them."""
    found = search(question)
    context = "\n\n".join(f"[Page {c['page']}] {c['text']}" for c in found)

    # STUDENT TODO (Level 3): if the best match is weak, skip the AI and report that no answer was found.
    #   Hint: if found[0]["score"] < 0.25: return "The answer could not be found in the document.", found

    messages = [
        {"role": "system", "content": ANSWER_RULES},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]
    demo = (f"**Demonstration mode:** the AI model is not connected, but retrieval completed. The most relevant "
            f"passage is on **page {found[0]['page']}**:\n\n> {found[0]['text'][:400]}...")
    return ask_llm(messages, temperature=TEMPERATURE, demo_reply=demo), found


# Quick test using the sample document.
print(build_knowledge_base(SAMPLE_PAGES, "Sample CS 101 syllabus"))
test_question = "What happens if I submit a lab late?"
reply, found = answer(test_question)
print(f"\nQuestion: {test_question}\nAnswer: {reply}\n\nPassages retrieved:")
for c in found:
    print(f"   [page {c['page']}, similarity {c['score']:.2f}] {c['text'][:90]}...")

## Step 4: Launch the application

Run this cell. Upload any text-based PDF (for example a syllabus, club regulations or a student handbook), or continue with the sample document. Re-run this cell after changing any cell above.

In [ ]:
import gradio as gr

gr.close_all()  # closes any previous version of the application that is still running


def on_upload(file):
    if file is None:
        return "No file uploaded."
    try:
        return build_knowledge_base(read_pdf(file), file.split("/")[-1])
    except Exception as e:
        raise gr.Error(f"The PDF could not be read: {e}")


def on_use_sample():
    return build_knowledge_base(SAMPLE_PAGES, "Sample CS 101 syllabus")


def on_ask(question):
    if not question.strip():
        raise gr.Error("Enter a question first.")
    reply, found = answer(question)
    # Show exactly what the AI received, so the answer can be checked.
    sources = "\n\n".join(f"**Page {c['page']}** (similarity {c['score']:.2f})\n> {c['text']}" for c in found)
    return reply, sources


with gr.Blocks(theme=gr.themes.Soft(), title=APP_TITLE) as demo:  # STUDENT TODO (Level 2): alternative theme: gr.themes.Glass()
    gr.Markdown(f"# {APP_TITLE}\n{APP_DESCRIPTION}")
    with gr.Row():
        pdf_input = gr.File(label="Upload a PDF", file_types=[".pdf"], type="filepath")
        with gr.Column():
            status_md = gr.Markdown(f"Current document: **{knowledge_base['name']}**")
            sample_btn = gr.Button("Use the sample syllabus")
    question_box = gr.Textbox(label="Question", placeholder="For example: How is the course graded?")
    ask_btn = gr.Button("Ask", variant="primary")
    # STUDENT TODO (Level 1): change the example questions to suit your document.
    gr.Examples(["How is the course graded?", "When are office hours?", "Can I use AI tools?",
                 "Who won the 2022 World Cup?"], inputs=question_box)
    answer_md = gr.Markdown()
    with gr.Accordion("Retrieved passages (the text given to the AI)", open=False):
        sources_md = gr.Markdown()

    pdf_input.upload(on_upload, inputs=pdf_input, outputs=status_md)
    sample_btn.click(on_use_sample, outputs=status_md)
    ask_btn.click(on_ask, inputs=question_box, outputs=[answer_md, sources_md])
    question_box.submit(on_ask, inputs=question_box, outputs=[answer_md, sources_md])

demo.launch(share=True)

## Level 3: Optional extensions

**1. Unanswerable questions (recommended for the demonstration).** Ask questions that are **not** covered by the document (the application includes "Who won the 2022 World Cup?" as an example). Check whether the application declines or invents an answer. Then remove the rule to use only the context from `ANSWER_RULES` and observe the effect.

**2. Chunk size experiment.** Test `CHUNK_SIZE = 40`, `120` and `400` with the same five questions. Record which setting gives the best answers in a table for the demonstration. Reload the document after each change.

**3. Similarity threshold.** Use the hint in `answer()` to skip the AI when no passage is sufficiently relevant. Use the similarity scores shown in the retrieved passages section to choose a suitable threshold.

**4. Multiple documents.** Allow several files to be uploaded at once (`gr.File(file_count="multiple")`) and include the file name in each citation.

**5. Conversation memory.** Replace the question field with `gr.ChatInterface` to support follow-up questions, such as "And what about the midterm?"

**6. Comparison without RAG.** Add a second answer generated by asking the AI the same question **without** the retrieved context. Show both answers side by side to demonstrate the value of RAG.

## Demonstration checklist
- [ ] A real PDF uploaded live, with three questions answered and correct page references
- [ ] The retrieved passages shown and used to explain how RAG works
- [ ] An unanswerable question that the application correctly declines to answer
- [ ] The results of one experiment (chunk size, number of chunks, or similarity threshold)

## Troubleshooting

| Issue | Resolution |
|---|---|
| `NameError: name '...' is not defined` | A cell was skipped. Run all cells above in order. |
| The application does not reflect recent changes | Re-run the edited cell **and** the "Launch the application" cell. |
| The notebook remains in demonstration mode | Add `GROQ_API_KEY` in Secrets, enable **Notebook access**, and re-run the setup cell. |
| "Usage limit reached" error | Wait about one minute. Free keys allow a limited number of requests per minute. |
| `SyntaxError` | Usually caused by a missing quotation mark `"`, bracket `)` or comma. Check the line shown in the error. |
| Unexpected behaviour | Select **Runtime → Restart session**, then run all cells from the top. |
| The public link no longer works | Links expire when the Colab session disconnects. Re-run the launch cell to create a new link. |